### ЗАДАЧА: Реестр пропусков коворкинга

Администратор коворкинга ведёт реестр пропусков резидентов.
Нужно собрать модель, которая позволит:
- загрузить пропуска в единый реестр,
- посмотреть только активные пропуска,
- отфильтровать резидентов по тарифу,
- посчитать суммарный остаток дней,
- понять, как меняется реестр после паузы, списания дня и продления.

В данных есть разные тарифы и статусы,
поэтому важно корректно валидировать plan, status и изменение состояния объекта.
        


In [62]:
# rows: pass_id|member_name|plan|days_left|status
rows = [
    'PS-100|Alice|flex|12|active',
    'PS-101|Bob|fixed|20|paused',
    'PS-102|Team Rocket|team|0|expired',
    'PS-103|Diana|flex|6|active',
]


class CoworkingPass:
    allowed_plans = {'flex', 'fixed', 'team'}
    allowed_statuses = {'active', 'paused', 'expired'}

    def __init__(self, pass_id, member_name, plan, days_left, status):
        # TODO: проверить plan и status, иначе raise ValueError(...)
        if plan not in self.allowed_plans:
            raise ValueError("Недопустимый план")
        if status not in self.allowed_statuses:
            raise ValueError("Недопустимый статус")
        # TODO: сохранить pass_id, member_name, plan, status
        self.pass_id = pass_id
        self.member_name = member_name
        self.plan = plan
        self.status = status
        # TODO: days_left хранить через внутреннее поле self._days_left
        self.days_left = days_left
        # TODO: значение days_left пропустить через property/setter
        

    @property
    def days_left(self):
        # TODO: вернуть текущее число оставшихся дней
        return self._days_left

    @days_left.setter
    def days_left(self, value):
        # TODO: привести value к int
        try:
            value = int(value)
        except ValueError:
            return "Ошибка, это не число"
        # TODO: если value < 0 -> raise ValueError('Days must be >= 0')
        if value < 0:
            raise ValueError("Days must be >= 0")
        # TODO: сохранить результат в self._days_left
        self._days_left = value

    def use_day(self):
        # TODO: если статус не 'active' -> raise ValueError(...)
        if self.status != "active":
            raise ValueError("Неверный статус")
        # TODO: если days_left == 0 -> raise ValueError(...)
        if self._days_left == 0:
            raise ValueError("Не хватает дней")
        # TODO: уменьшить days_left на 1
        self._days_left -= 1
        # TODO: если после списания days_left == 0, перевести статус в 'expired'
        if self._days_left == 0:
            self.status = "expired"

    def pause(self):
        # TODO: если статус 'expired' -> raise ValueError(...)
        if self.status == "expired":
            raise ValueError("Статус истек")
        # TODO: перевести пропуск в 'paused'
        self.status = "paused"


    def resume(self):
        # TODO: если days_left == 0 -> raise ValueError(...)
        if self._days_left == 0:
            raise ValueError("Истекло время")
        # TODO: перевести пропуск в 'active'
        self.status = "active"


    def renew(self, extra_days):
        # TODO: привести extra_days к int
        try:
            extra_days = int(extra_days)
        except:
            raise ValueError("Дни должны быть числом")
        # TODO: если extra_days <= 0 -> raise ValueError(...)
        if extra_days <= 0:
            raise ValueError("Дни должны быть положительными")
        # TODO: увеличить days_left
        self._days_left += extra_days
        # TODO: если days_left > 0 и статус был 'expired', перевести в 'active'
        if self._days_left > 0 and self.status == "expired":
            self.status = "active"

    @classmethod
    def from_row(cls, row):
        # TODO: split по '|'
        parts = row.split("|")
        # TODO: ожидать 5 частей: pass_id, member_name, plan, days_left, status
        if len(parts) != 5:
            raise ValueError("Количество частей должно состоять из 5")
        pass_id, member_name, plan, _days_left, status = parts
        # TODO: вернуть CoworkingPass(...)
        return CoworkingPass(pass_id, member_name, plan, _days_left, status)

    def __repr__(self):
        # TODO: вернуть строку вида CoworkingPass(pass_id='...', member_name='...', status='...', days_left=...)
        return f"CoworkingPass(pass_id='{self.pass_id}', member_name='{self.member_name}', status='{self.status}', days_left='{self.days_left}')"


class CoworkingRegistry:
    def __init__(self):
        self.items = []

    def add(self, coworking_pass):
        # TODO: добавить объект в self.items
        self.items.append(coworking_pass)

    def load(self, rows):
        # TODO: для каждой строки создать CoworkingPass.from_row(row)
        for row in rows:
            row = CoworkingPass.from_row(row)
        # TODO: добавить объект в реестр через add(...)
            self.add(row)
        

    def active_passes(self):
        # TODO: вернуть список пропусков со статусом 'active'
        return [item for item in self.items if item.status == "active"]

    def by_plan(self, plan):
        # TODO: вернуть список пропусков нужного тарифа
        return [item for item in self.items if item.plan == plan]

    def total_days_left(self):
        # TODO: вернуть суммарное число оставшихся дней
        return sum(item.days_left for item in self.items)

    def status_summary(self):
        # TODO: собрать dict вида status -> count
        status_summary = {}
        for item in self.items:
            status_summary[item.status] = status_summary.get(item.status, 0)
            status_summary[item.status] += 1
        return status_summary
  
    def find(self, pass_id):
        # TODO: вернуть пропуск по pass_id или None
        for item in self.items:
            if item.pass_id == pass_id:
                return item
        return None

    def largest_balance(self):
        # TODO: найти пропуск с максимальным days_left
        largest_balance = {}
        for item in self.items:
            largest_balance[item.pass_id] = largest_balance.get(item.pass_id, item.days_left)
        # TODO: вернуть tuple(pass_id, days_left)   
        return max(largest_balance.items(), key=lambda x: x[1])


registry = CoworkingRegistry()

# TODO: загрузить rows в registry
registry.load(rows)
# TODO: вывести все пропуска
print("Все пропуска:")
for item in registry.items:
    print(item)
# TODO: вывести active_passes()
print("Все активные пропуска:")
for item in registry.active_passes():
    print(item)
# TODO: вывести by_plan('flex')
print("Все клиенты с тарифом 'flex':")
for item in registry.by_plan('flex'):
    print(item)
# TODO: вывести total_days_left()
print(f"Общее суммарное количестов дней = {registry.total_days_left()}.")
# TODO: вывести status_summary()
status_summary = registry.status_summary()
print("Суммарное количество статусов:")
for status, count in status_summary.items():
    print(f"Статус '{status}' встречается {count} раз")
# TODO: найти пропуск 'PS-101', возобновить его и вывести status_summary()
find_101 = registry.find("PS-101")
if find_101:
    find_101.resume()
else:
    print("Такой id не найден")
status_summary_new = registry.status_summary()
print("После активации id 'PS-101' количество статусов:")
for status, count in status_summary_new.items():
    print(f"Статус '{status}' встречается {count} раз")
# TODO: найти пропуск 'PS-100', списать один день и вывести объект
find_100 = registry.find("PS-100")
if find_100:
    find_100.use_day()
else:
    print("Такой id не найден")
print("После списания 1 дня у id 'PS-100':")
print(find_100)
# TODO: найти пропуск 'PS-102', продлить на 5 дней и вывести объект
find_102 = registry.find("PS-102")
if find_102:
    find_102.renew(5)
else:
    print("Такой id не найден")
print("После продления на 5 дней у id 'PS-102':")    
print(find_102)
# TODO: вывести largest_balance()
largest_balance = registry.largest_balance()
id = largest_balance[0]
days = largest_balance[1]
print(f"У id '{largest_balance[0]}' больше всего дней = {largest_balance[1]}.")
        


Все пропуска:
CoworkingPass(pass_id='PS-100', member_name='Alice', status='active', days_left='12')
CoworkingPass(pass_id='PS-101', member_name='Bob', status='paused', days_left='20')
CoworkingPass(pass_id='PS-102', member_name='Team Rocket', status='expired', days_left='0')
CoworkingPass(pass_id='PS-103', member_name='Diana', status='active', days_left='6')
Все активные пропуска:
CoworkingPass(pass_id='PS-100', member_name='Alice', status='active', days_left='12')
CoworkingPass(pass_id='PS-103', member_name='Diana', status='active', days_left='6')
Все клиенты с тарифом 'flex':
CoworkingPass(pass_id='PS-100', member_name='Alice', status='active', days_left='12')
CoworkingPass(pass_id='PS-103', member_name='Diana', status='active', days_left='6')
Общее суммарное количестов дней = 38.
Суммарное количество статусов:
Статус 'active' встречается 2 раз
Статус 'paused' встречается 1 раз
Статус 'expired' встречается 1 раз
После активации id 'PS-101' количество статусов:
Статус 'active' встреча